# Modeling and Final Selection

This notebook compares Linear Regression, Decision Tree, and Random Forest using the same preprocessing and 10-fold CV strategy as the project scripts.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [ ]:
data_path = Path('..') / 'data' / 'housing.csv'
housing = pd.read_csv(data_path)
housing['income_cat'] = pd.cut(
    housing['median_income'],
    bins=[0.0, 1.5, 3.0, 4.5, 6.0, float('inf')],
    labels=[1, 2, 3, 4, 5]
)
housing.head()

In [ ]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(housing, housing['income_cat']):
    train_set = housing.loc[train_idx].drop('income_cat', axis=1).copy()
    test_set = housing.loc[test_idx].drop('income_cat', axis=1).copy()

train_features = train_set.drop('median_house_value', axis=1)
train_labels = train_set['median_house_value'].copy()

num_attributes = train_features.drop('ocean_proximity', axis=1).columns.tolist()
cat_attributes = ['ocean_proximity']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder()),
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attributes),
    ('cat', cat_pipeline, cat_attributes),
])

train_prepared = full_pipeline.fit_transform(train_features)
train_prepared.shape

In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'RandomForestRegressor': RandomForestRegressor(),
}

rows = []
for name, model in models.items():
    mse_scores = cross_val_score(
        model,
        train_prepared,
        train_labels,
        scoring='neg_mean_squared_error',
        cv=10,
    )
    mae_scores = cross_val_score(
        model,
        train_prepared,
        train_labels,
        scoring='neg_mean_absolute_error',
        cv=10,
    )

    rows.append({
        'model': name,
        'rmse': float(np.mean(np.sqrt(-mse_scores))),
        'mae': float(np.mean(-mae_scores)),
    })

results = pd.DataFrame(rows).sort_values(by='rmse')
results

In [ ]:
best_model_name = results.iloc[0]['model']
best_model_name